# Multi-Series + Covariates: Store-Level Demand Planning
# 多序列 + 协变量：门店级需求计划

This is a common production pattern: many related stores/SKUs, future-known promotions and holidays, historical-only weather/stockout signals, and per-series forecasts.

这是常见生产模式：多门店/SKU，未来已知促销和节假日，历史天气/缺货信号，以及每序列预测。

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)

def make_retail_demand(n_days=240, n_stores=1, start="2023-01-01"):
    rows = []
    for i in range(n_stores):
        rng = np.random.default_rng(100 + i)
        dates = pd.date_range(start, periods=n_days, freq="D")
        dow = dates.dayofweek.to_numpy()
        month = dates.month.to_numpy()
        holiday = ((dow >= 5) | rng.binomial(1, 0.04, n_days).astype(bool)).astype(int)
        promotion = rng.binomial(1, 0.14 + 0.06 * (dow >= 4), n_days).astype(int)
        price_index = 1.0 + 0.04 * np.sin(np.linspace(0, 5 * np.pi, n_days)) + rng.normal(0, 0.015, n_days)
        temperature = 18 + 10 * np.sin(np.linspace(-0.8, 2.8 * np.pi, n_days)) + rng.normal(0, 1.8, n_days)
        stockout = rng.binomial(1, 0.025, n_days)
        baseline = 120 + 18 * i
        weekly = np.where(dow < 5, 8, 28)
        seasonal = 16 * np.sin(2 * np.pi * np.arange(n_days) / 365.25 + i / 3)
        trend = 0.08 * np.arange(n_days)
        demand = (
            baseline + weekly + seasonal + trend
            + 34 * promotion + 22 * holiday
            + 0.9 * np.maximum(temperature - 20, 0)
            - 75 * (price_index - 1.0)
            - 45 * stockout
            + rng.normal(0, 7, n_days)
        )
        rows.append(pd.DataFrame({
            "date": dates,
            "store_id": f"store_{i + 1:02d}",
            "sales": np.maximum(demand, 1),
            "promotion": promotion,
            "holiday": holiday,
            "price_index": price_index,
            "temperature": temperature,
            "stockout": stockout,
            "month": month,
        }))
    return pd.concat(rows, ignore_index=True)

In [ ]:
panel = make_retail_demand(n_days=220, n_stores=4)
train = panel.groupby("store_id", group_keys=False).head(200).reset_index(drop=True)
valid = panel.groupby("store_id", group_keys=False).tail(20).reset_index(drop=True)
future_covariates = valid[["date", "store_id", "promotion", "holiday"]].reset_index(drop=True)

print(panel.groupby("store_id").size())
panel.head()

In [ ]:
from PipelineTS.plot import plot_series

plot_series(panel, time_col="date", target_col="sales", id_col="store_id", title="Store-level demand", lang="zh")

In [ ]:
from PipelineTS.pipeline import ModelPipeline

panel_pipe = ModelPipeline(
    time_col="date",
    target_col="sales",
    id_col="store_id",
    lags=14,
    known_covariates=["promotion", "holiday"],
    past_covariates=["temperature", "price_index", "stockout"],
    include_models=["random_forest", "extra_forest"],
    quantile=0.9,
    cv=2,
    random_forest__n_estimators=100,
    extra_forest__n_estimators=100,
)
panel_lb = panel_pipe.fit(train, valid_data=valid)
panel_lb

In [ ]:
panel_pred = panel_pipe.predict(20, future_covariates=future_covariates)
print(panel_pred.groupby("store_id").size())
panel_pred.head(10)

In [ ]:
panel_q = panel_pipe.predict_quantiles(20, levels=[0.8, 0.9], future_covariates=future_covariates)
panel_q.head()

In [ ]:
from PipelineTS.pipeline import SmartRouter

router_panel = SmartRouter(
    time_col="date",
    target_col="sales",
    id_col="store_id",
    known_covariates=["promotion", "holiday"],
    past_covariates=["temperature", "price_index", "stockout"],
    preset="fast",
    include_models=["random_forest", "extra_forest", "multi_output_model"],
    quantile=0.9,
    time_limit=90,
)
router_panel.fit(train, valid_data=valid)
router_panel.predict(20, future_covariates=future_covariates).head()

In [ ]:
print("Profile series count:", router_panel.profile_.n_series)
print("Data insights:")
router_panel.insights_.summary()